# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*
Building the feature vector: We extract raw performance indicators, compute engineered ratios (like click-to-impression CTR and content age), handle nulls with median/zero fills, and assemble a clean feature matrix ready for modeling.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np
import urllib.request

file_path = "content_refresh_anonymized.csv"
data_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

if not os.path.exists(file_path):
    urllib.request.urlretrieve(data_url, file_path)

df = pd.read_csv(file_path)

# Build feature engineering pipeline
features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan)

# Fill missing values safely
X["content_age_days"] = X["content_age_days"].fillna(X["content_age_days"].median())
X["days_since_last_update"] = X["days_since_last_update"].fillna(X["days_since_last_update"].median())
X = X.fillna(0)

print(f"Feature vector shape built successfully: {X.shape}")
display(X.head(3))

Feature vector shape built successfully: (30000, 6)


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count
0,187,20,3803,10.6,0.76,3221.0
1,445,25,15320,20.3,0.05,2481.0
2,141,20,12581,36.5,0.09,3515.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*
- content_age_days: Total age since publication. Missing filled with median. Available prior to decision moment.
- days_since_last_update: Time elapsed since the last content update. Missing filled with median. Available prior to decision.
- impressions_90d: Total search impressions over the past 90 days. Missing filled with zero. Available prior to decision.
- avg_position: Average ranking position in search results. Missing filled with zero. Available prior to decision.
- ctr: Click-through rate. Missing filled with zero. Available prior to decision.
- word_count: Total word count of the page. Missing filled with median. Available prior to decision.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Summary statistics of the built feature vector
display(X.describe())

,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count
count,30000.00000,30000.000000,30000.000000,30000.00000,30000.000000,30000.000000
mean,256.16780,46.098300,5200.366300,16.34238,0.510733,2310.205433
std,132.70793,42.078709,16838.019547,15.21679,3.279162,1846.788556
min,90.00000,1.000000,1.000000,0.00000,0.000000,0.000000
25%,132.00000,20.000000,81.000000,6.20000,0.000000,0.000000
50%,236.00000,20.000000,731.000000,10.80000,0.070000,2605.000000
75%,333.00000,104.000000,3615.250000,22.30000,0.290000,3247.000000
max,564.00000,373.000000,517715.000000,245.00000,100.000000,9546.000000


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*
Leakage Hunt: We intentionally inject a target-derived column (trend_pct) into the feature set to demonstrate how an accidental leak artificially inflates model performance, then we isolate and remove it to maintain data integrity.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier

y = (df["trend_direction"].str.lower() == "down").astype(int)

# Clean model score
tree_clean = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X, y)
clean_score = tree_clean.score(X, y)

# Leaky feature injection test
X_leaky = X.copy()
X_leaky["trend_pct_leak"] = df["trend_pct"]
tree_leaky = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_leaky, y)
leaky_score = tree_leaky.score(X_leaky, y)

print(f"Clean Model Accuracy (Honest): {clean_score:.3f}")
print(f"Leaky Model Accuracy (Compromised): {leaky_score:.3f} -> Leakage detected and prevented!")

Clean Model Accuracy (Honest): 0.637
Leaky Model Accuracy (Compromised): 1.000 -> Leakage detected and prevented!


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
- trend_pct: Excluded because it is mathematically derived from the exact target outcome percentage (direct data leakage).
- internal_health_score: Excluded because it represents a proprietary post-decision product flag rather than a raw observable input.
- client_domain_name: Excluded to completely preserve client privacy and prevent overfitting to specific domains.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = ["trend_pct", "internal_health_score", "client_domain_name"]
print(f"Successfully excluded fields to protect privacy and prevent leakage: {excluded_fields}")

Successfully excluded fields to protect privacy and prevent leakage: ['trend_pct', 'internal_health_score', 'client_domain_name']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.